In [2]:
import pandas as pd
import numpy as np

from xgboost import XGBRegressor

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [3]:
df = pd.read_excel("DF_FINAL.xlsx")

df.head()

calles_base = [
        "Avenida Cardenal Herrera Oria",
        "Avenida Alfonso XIII",
        "Avenida Brasilia",
        "Avenida General Perón",
        "Calle Arturo Soria",
        "Calle Costa Rica",
        "Calle Génova",
        "Calle Hermanos García Noblejas",
        "Calle José Abascal",
        "Calle López de Hoyos",
        "Calle Príncipe de Vergara",
        "Calle Serrano",
        "Calle Velázquez",
        "Paseo de la Castellana"
]

calles_noreste = []

for base in calles_base:
    for col in df.columns:
        if base in col:
            calles_noreste.append(col)

L = 3  # número de lags

X = pd.DataFrame()

for street in calles_noreste:
    for lag in range(1, L + 1):
        X[f"{street}_t-{lag}"] = df[street].shift(lag)

# Variables externas reales
X["AEMET_tmed"] = df["AEMET_tmed"]
X["AEMET_prec"] = df["AEMET_prec"]
X["dia_semana"] = df["dia_semana"]
X["festivo"] = df["festivo"]

# Convertir variables categóricas a numéricas
X["dia_semana"] = X["dia_semana"].map({
    "lunes": 0,
    "martes": 1,
    "miércoles": 2,
    "jueves": 3,
    "viernes": 4,
    "sábado": 5,
    "domingo": 6
})

X["festivo"] = X["festivo"].map({
    "no": 0,
    "sí": 1
})


y_col = [col for col in df.columns if "Paseo de la Castellana" in col][0]
y = df[y_col]

# Limpiar NaNs
X = X.dropna()
y = y.loc[X.index]



In [5]:
# División temporal (80% train, 20% test)
split = int(len(X) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

print(X_train.shape, X_test.shape)

(15376, 79) (3845, 79)


In [14]:
with pd.ExcelWriter("dataset_XGBoost_Castellana.xlsx") as writer:
    X_train.to_excel(writer, sheet_name="X_train")
    X_test.to_excel(writer, sheet_name="X_test")
    y_train.to_excel(writer, sheet_name="y_train")
    y_test.to_excel(writer, sheet_name="y_test")

In [7]:
from xgboost import XGBRegressor

model = XGBRegressor(
    n_estimators=400,
    max_depth=5,
    learning_rate=0.03,
    subsample=0.9,
    colsample_bytree=0.9,
    gamma=0.1,
    random_state=42
)

model.fit(X_train, y_train)

XGBRegressor(base_score=None, booster=None, callbacks=None,
             colsample_bylevel=None, colsample_bynode=None,
             colsample_bytree=0.9, device=None, early_stopping_rounds=None,
             enable_categorical=False, eval_metric=None, feature_types=None,
             feature_weights=None, gamma=0.1, grow_policy=None,
             importance_type=None, interaction_constraints=None,
             learning_rate=0.03, max_bin=None, max_cat_threshold=None,
             max_cat_to_onehot=None, max_delta_step=None, max_depth=5,
             max_leaves=None, min_child_weight=None, missing=nan,
             monotone_constraints=None, multi_strategy=None, n_estimators=400,
             n_jobs=None, num_parallel_tree=None, ...)

In [8]:
y_pred = model.predict(X_test)
# Predicción próxima hora
X_next = X.iloc[[-1]]
y_next = model.predict(X_next)

print("Predicción tráfico en Castellana (N-S) próxima hora:", y_next[0])

Predicción tráfico en Castellana (N-S) próxima hora: 314.25595


In [9]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

mask = y_test != 0
mape = np.mean(np.abs((y_test[mask] - y_pred[mask]) / y_test[mask])) * 100

print(f"MAE: {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"MAPE: {mape:.2f}%")

MAE: 67.13
RMSE: 96.14
MAPE: 12.44%


In [10]:
importances = pd.Series(
    model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

print(importances.head(10))

Calle Hermanos García Noblejas_S-N_t-1    0.574693
Paseo de la Castellana_N-S_t-1            0.100271
Calle Costa Rica_E-O_t-1                  0.085286
Calle Arturo Soria_S-N_t-1                0.066753
Calle Arturo Soria_N-S_t-3                0.015547
Calle Costa Rica_O-E_t-2                  0.009294
Calle Velázquez_S-N_t-1                   0.008674
Calle Príncipe de Vergara_S-N_t-2         0.007002
Calle Arturo Soria_N-S_t-1                0.005342
Paseo de la Castellana_N-S_t-2            0.005066
dtype: float32


In [11]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=10,
    random_state=0,
    n_jobs=-1,
    scoring="neg_mean_absolute_error"
)

perm_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance": perm.importances_mean
}).sort_values("importance", ascending=False)

perm_df.head(10)


,feature,importance
69,Paseo de la Castellana_N-S_t-1,205.654187
45,Calle Hermanos García Noblejas_S-N_t-1,85.502455
30,Calle Costa Rica_E-O_t-1,33.261159
32,Calle Costa Rica_E-O_t-3,16.368193
0,Avenida Cardenal Herrera Oria_E-O_t-1,7.159218
31,Calle Costa Rica_E-O_t-2,6.061761
27,Calle Arturo Soria_S-N_t-1,4.838331
34,Calle Costa Rica_O-E_t-2,4.793925
77,dia_semana,2.880534
47,Calle Hermanos García Noblejas_S-N_t-3,2.837673


In [12]:
top_k = 8
top_features = perm_df.head(top_k)
top_features


,feature,importance
69,Paseo de la Castellana_N-S_t-1,205.654187
45,Calle Hermanos García Noblejas_S-N_t-1,85.502455
30,Calle Costa Rica_E-O_t-1,33.261159
32,Calle Costa Rica_E-O_t-3,16.368193
0,Avenida Cardenal Herrera Oria_E-O_t-1,7.159218
31,Calle Costa Rica_E-O_t-2,6.061761
27,Calle Arturo Soria_S-N_t-1,4.838331
34,Calle Costa Rica_O-E_t-2,4.793925


In [13]:
top_streets = (
    top_features["feature"]
    .str.split("_")
    .str[0]
    .value_counts()
)
top_streets

feature
Calle Costa Rica                  4
Paseo de la Castellana            1
Calle Hermanos García Noblejas    1
Avenida Cardenal Herrera Oria     1
Calle Arturo Soria                1
Name: count, dtype: int64